# 🍎 Apple Game RL Training

| 단계 | 위치 | 내용 |
|------|------|------|
| **Phase 1** | 로컬 PC (Go) | `sl_data.bin` 생성 |
| **Phase 2** | 이 노트북 | SL 학습 → `model_sl.onnx` |
| **Phase 3** | 이 노트북 | PPO RL → `model_rl.onnx` |
| **Phase 4** | 로컬 PC (Go) | ModelAnA vs AnA 비교 |

### 사전 준비
1. 로컬에서 데이터 생성: `go run . -gen-sl 500 -sl-out sl_data.bin`
2. `sl_data.bin` (~228 MB) → Google Drive `내 드라이브/apple_game/` 에 업로드
3. **런타임 → 런타임 유형 변경 → T4 GPU**
4. 설정 셀의 `GITHUB_URL` 확인 후 위에서부터 순서대로 실행

In [ ]:
# ── 설정 ────────────────────────────────────────────────────────────────────────
GITHUB_URL   = 'https://github.com/scheveningen361/10_apple_game.git'
DRIVE_FOLDER = '/content/drive/MyDrive/apple_game'
SL_EPOCHS    = 50
SL_BATCH     = 2048
RL_ITERS     = 200
RL_EPISODES  = 200

In [ ]:
# ── 1. GPU 확인 ─────────────────────────────────────────────────────────────────
import torch, os
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
else:
    print('⚠️  GPU 없음 — 런타임 유형을 T4로 변경하세요')

In [ ]:
# ── 2. Drive 마운트 ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_FOLDER, exist_ok=True)
print('Drive 마운트 완료:', DRIVE_FOLDER)

In [ ]:
# ── 3. repo 클론 / 업데이트 + 코드 버전 검증 ───────────────────────────────────
import shutil
REPO_DIR = '/content/apple_game'
RL_DIR   = REPO_DIR + '/rl'

if os.path.exists(REPO_DIR + '/.git'):
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_URL} {REPO_DIR}

os.chdir(RL_DIR)

# 현재 커밋 확인
print('\n--- 현재 커밋 ---')
!cd {REPO_DIR} && git log --oneline -3

# train_sl.py 버전 확인 (load_sl_tensors 함수가 있으면 최신)
print('\n--- train_sl.py 버전 확인 ---')
!grep -n 'load_sl_tensors\|_tick\|uint8' {RL_DIR}/train_sl.py | head -5

print('\n작업 디렉토리:', os.getcwd())
!ls -lh

In [ ]:
# ── 4. sl_data.bin 복사 ─────────────────────────────────────────────────────────
src = f'{DRIVE_FOLDER}/sl_data.bin'
dst = f'{RL_DIR}/sl_data.bin'

if os.path.exists(dst):
    print(f'✅ sl_data.bin 이미 있음 ({os.path.getsize(dst)/1e6:.0f} MB)')
elif os.path.exists(src):
    shutil.copy(src, dst)
    print(f'✅ Drive에서 복사 완료 ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    raise FileNotFoundError(f'sl_data.bin 없음! Drive의 {DRIVE_FOLDER}/ 에 업로드하세요')

---
## Phase 2: SL 학습

In [ ]:
# ── Phase 2: SL 학습 ────────────────────────────────────────────────────────────
!python train_sl.py \
    --data   sl_data.bin \
    --out    model_sl.pt \
    --epochs {SL_EPOCHS} \
    --batch  {SL_BATCH}

In [ ]:
# ── SL 체크포인트 확인 ──────────────────────────────────────────────────────────
ckpt = torch.load('model_sl.pt', map_location='cpu')
print(f"Best val RMSE : {ckpt['best_val'] * 170:.4f}")
print(f"Epoch         : {ckpt['epoch'] + 1}")
print(f"Config        : {ckpt['config']}")

In [ ]:
# ── SL → ONNX 변환 ─────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, RL_DIR)
from train_sl import AppleNetSL, export_onnx

ckpt  = torch.load('model_sl.pt', map_location='cpu')
cfg   = ckpt['config']
model = AppleNetSL(channels=cfg['channels'], n_blocks=cfg['blocks'])
model.load_state_dict(ckpt['model'])
export_onnx(model, 'model_sl.onnx', device='cpu')
print(f"model_sl.onnx  {os.path.getsize('model_sl.onnx')/1e6:.1f} MB")

In [ ]:
# ── SL 모델 Drive 백업 ─────────────────────────────────────────────────────────
for f in ['model_sl.pt', 'model_sl.onnx']:
    if os.path.exists(f):
        shutil.copy(f, f'{DRIVE_FOLDER}/{f}')
        print(f'💾 {DRIVE_FOLDER}/{f}')

---
## Phase 3: PPO RL 학습

In [ ]:
# ── Phase 3: SB3 설치 + PPO RL 학습 ──────────────────────────────────────
!pip install -q stable-baselines3 sb3-contrib gymnasium

!python train_rl.py \n    --sl       model_sl.pt \n    --out      model_rl.zip \n    --iters    {RL_ITERS}   \n    --n-envs   4            \n    --n-steps  2048

In [ ]:
# ── RL 체크포인트 확인 ──────────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO

rl_model = MaskablePPO.load('model_rl.zip')
print(f'Policy device  : {rl_model.device}')
print(f'Total timesteps: {rl_model.num_timesteps:,}')
print('RL 모델 로드 OK')

In [ ]:
# ── RL → ONNX 변환 ─────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, RL_DIR)
from train_rl import export_onnx_value

export_onnx_value('model_rl.zip', 'model_rl.onnx')
print(f'model_rl.onnx  {os.path.getsize("model_rl.onnx")/1e6:.1f} MB')

In [ ]:
# ── 최종 백업 ──────────────────────────────────────────────────────────────────
for f in ['model_rl.zip', 'model_rl.onnx']:
    if os.path.exists(f):
        shutil.copy(f, f'{DRIVE_FOLDER}/{f}')
        print(f'💾 {DRIVE_FOLDER}/{f}')
!ls -lh {DRIVE_FOLDER}

---
## Phase 4: 로컬 평가

Drive에서 `model_rl.onnx` 다운로드 후:
```bash
go run -tags nn . -nn-ana -n 100 -model model_rl.onnx -ort-lib onnxruntime.dll
```